# Battery Swelling Prediction Workflow

This notebook records the end-to-end workflow for:

1. dataset cleaning / subset preparation
2. frequency-domain ECM fitting
3. time-domain ECM fitting
4. model training
5. evaluation and plotting

It is designed to reuse the existing command-line scripts in this repo, while making the run history and outputs easier to inspect in one place.

## 0. Manual configuration

Set the output folder and dataset paths here before running anything else.

In [ ]:
from pathlib import Path
import os
import subprocess
import shlex
import json
import shutil
import pandas as pd

REPO_ROOT = Path.cwd()

# Support running the notebook from the repo root, notebooks/, or src/.
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

VENV_PYTHON = REPO_ROOT / ".venv" / "bin" / "python"

# -------------------------
# Manual run configuration
# -------------------------
RUN_NAME = "latest_test_26_5_20_1_repro"
OUTPUT_ROOT = REPO_ROOT / "data" / RUN_NAME
LOG_DIR = OUTPUT_ROOT / "logs"

# Generic dataset paths
WORKBOOK_XLSX_DIR = REPO_ROOT / "dataset" / "udc_xlsx"
RAW_DATA_DIR = REPO_ROOT / "dataset" / "raw_data"

# Example real local paths for this repo (edit as needed)
WORKBOOK_XLSX_DIR = REPO_ROOT / "dataset" / "OneDrive_1_2-20-2026" / "HYCL"
RAW_DATA_DIR = REPO_ROOT / "dataset" / "raw_data" / "HYCL"

# -------------------------------------------------
# Frequency-domain ECM fitting control
# -------------------------------------------------
# If True, rerun frequency-domain ECM fitting using the same pre15 seed subset used in 14_3.
# If False, skip fitting and reuse the previously generated td-compatible ECM directory from 14_3.
RERUN_FREQUENCY_ECM_FIT = False

# Exact frequency-domain seed workbook subset used in the 14_3 -> 14_6 chain.
WORKBOOK_PRIOR_SEED_DIR = REPO_ROOT / "data" / "latest_test_26_5_14_3" / "xlsx_prior_subset_pre15"

# Reuse the same td-compatible frequency-domain ECM results used in 14_3 by default.
EXISTING_ECM_DIR = REPO_ROOT / "data" / "latest_test_26_5_14_3" / "ecm_w_cycle_td_compatible_pre15"

# Output location if you choose to rerun td-compatible frequency ECM fitting.
ECM_OUT_DIR = OUTPUT_ROOT / "ecm_w_cycle_td_compatible_pre15"
ECM_SOURCE_DIR = ECM_OUT_DIR if RERUN_FREQUENCY_ECM_FIT else EXISTING_ECM_DIR

PRIOR_CSV = OUTPUT_ROOT / "eis_time_domain_priors_td_compatible.csv"
RAW_SUBSET_DIR = OUTPUT_ROOT / "raw_subset_hycl_one_per_serial"
RAW_FEATURE_CSV = OUTPUT_ROOT / "device_ecm_with_priors_global.csv"
BASE_FEATURE_TABLE = REPO_ROOT / "data" / "latest_test_26_4_21" / "feature_table_ecm_complete.csv"
MERGED_FEATURE_TABLE_ALL = OUTPUT_ROOT / "feature_table_ecm_complete_with_device_global.csv"
MERGED_FEATURE_TABLE = OUTPUT_ROOT / "feature_table_hycl_only_with_device_global.csv"
TRAIN_TABLE = OUTPUT_ROOT / "feature_table_hycl_only_td_replaced.csv"
MODEL_OUT_DIR = OUTPUT_ROOT / "results_xgb_current_hycl_only"
RESULT_RUN_TAG = "raw_hycl_td_replaced_xgb"

# Time-domain segment selection controls.
# Use last/longest to reproduce the older 14_6 behavior, or best/best_td
# to favor segments with more usable points for the constrained TD fitter.
TD_CYCLE_MODE = "best"
TD_SEGMENT_CHOICE = "best_td"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
print("repo_root =", REPO_ROOT)
print("venv_python =", VENV_PYTHON)
print("rerun_frequency_ecm_fit =", RERUN_FREQUENCY_ECM_FIT)
print("workbook_prior_seed_dir =", WORKBOOK_PRIOR_SEED_DIR)
print("ecm_source_dir =", ECM_SOURCE_DIR)
print("base_feature_table =", BASE_FEATURE_TABLE)
print("td_cycle_mode =", TD_CYCLE_MODE)
print("td_segment_choice =", TD_SEGMENT_CHOICE)
OUTPUT_ROOT


In [ ]:
def run(cmd, log_name=None, cwd=REPO_ROOT, check=True):
    print(cmd)
    if log_name is not None:
        log_path = LOG_DIR / log_name
        with open(log_path, "w", encoding="utf-8") as f:
            proc = subprocess.run(cmd, shell=True, cwd=cwd, stdout=f, stderr=subprocess.STDOUT, text=True)
        print(f"[log] {log_path}")
    else:
        proc = subprocess.run(cmd, shell=True, cwd=cwd, text=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with code {proc.returncode}: {cmd}")
    return proc.returncode


def show_log(log_name, n=200):
    p = LOG_DIR / log_name
    text = p.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()
    print("\n".join(lines[-n:]))


## 1. Dataset cleaning / subset preparation

This section checks the workbook dataset and raw dataset, and optionally builds a raw subset that is easier to process.

Typical reasons to subset raw data:

- one representative file per serial
- only serials that appear in the labeled workbook-derived feature table
- only files that overlap with ECM priors

In [ ]:
print('Workbook dir:', WORKBOOK_XLSX_DIR)
print('Raw dir:', RAW_DATA_DIR)
print('Workbook exists:', WORKBOOK_XLSX_DIR.exists())
print('Raw exists:', RAW_DATA_DIR.exists())

xlsx_files = sorted(WORKBOOK_XLSX_DIR.rglob('*.xlsx')) if WORKBOOK_XLSX_DIR.exists() else []
raw_files = sorted([p for p in RAW_DATA_DIR.rglob('*') if p.is_file()]) if RAW_DATA_DIR.exists() else []
print('n workbook files =', len(xlsx_files))
print('n raw files =', len(raw_files))

### 1a. Reproduce the exact raw subset used in `latest_test_26_5_14_3`

For the `14_3 -> 14_6` chain, the raw subset was **not** an arbitrary one-file-per-serial sample.
It was restricted to HYCL serials present in:

- `data/latest_test_26_4_21/feature_table_ecm_complete.csv`

Then one representative raw file was kept per matching serial, yielding the same
`raw_subset_hycl_one_per_serial` used to build the merged table later filtered in `14_6`.


In [ ]:
cmd = f'''{VENV_PYTHON} - <<"PY"
import sys, shutil
from pathlib import Path
import pandas as pd
sys.path.insert(0, 'src')
from parse_raw_maccor import parse_metadata

ft = pd.read_csv(r"{BASE_FEATURE_TABLE}")
serials = set(ft.loc[ft['group_tag']=='HYCL','serial'].astype(str).str.strip().str.upper())
raw_dir = Path(r"{RAW_DATA_DIR}")
out_dir = Path(r"{RAW_SUBSET_DIR}")
out_dir.mkdir(parents=True, exist_ok=True)
by_serial = {{}}
for p in sorted(raw_dir.iterdir()):
    if not p.is_file() or p.name.startswith('.'):
        continue
    serial = str(parse_metadata('', p.name).get('serial','')).strip().upper()
    if serial in serials:
        by_serial[serial] = p
for serial, p in sorted(by_serial.items()):
    shutil.copy2(p, out_dir / p.name)
print('copied_files', len(by_serial))
print('copied_serials', len(by_serial))
PY'''
run(cmd, log_name='01_make_raw_subset.log')
show_log('01_make_raw_subset.log', n=80)


## 2. Frequency-domain ECM fitting

This notebook follows the same frequency-domain branch used upstream of `latest_test_26_5_14_6`:

- seed workbooks: `data/latest_test_26_5_14_3/xlsx_prior_subset_pre15`
- circuit family: `td_compatible`
- sheet: `02_PreEIS`
- warburg: `none`

You can either:

- rerun fitting from the seed subset (`RERUN_FREQUENCY_ECM_FIT = True`), or
- reuse the existing `14_3` result directory (`RERUN_FREQUENCY_ECM_FIT = False`)


In [ ]:
if RERUN_FREQUENCY_ECM_FIT:
    cmd = f'''MPLCONFIGDIR="{OUTPUT_ROOT / '.mplconfig'}" {VENV_PYTHON} src/ecm_fit.py \
  --xlsx_dir "{WORKBOOK_PRIOR_SEED_DIR}" \
  --recursive \
  --sheet 02_PreEIS \
  --fit_mode best_block \
  --circuit_family td_compatible \
  --warburg none \
  --out_dir "{ECM_OUT_DIR}" \
  --log_file "{LOG_DIR / '02_ecm_fit.internal.log'}"'''
    run(cmd, log_name='02_ecm_fit.log')
    show_log('02_ecm_fit.log', n=120)
else:
    print('Skipping frequency-domain ECM fitting and reusing existing results:')
    print(ECM_SOURCE_DIR)
    print('Set RERUN_FREQUENCY_ECM_FIT = True to rerun fitting from the same pre15 seed subset.')


## 3. Convert frequency-domain ECM into time-domain priors

This step converts frequency-domain ECM results into:

- initial guesses
- lower bounds
- upper bounds

for the time-domain fitter.

In [ ]:

cmd = f'''{VENV_PYTHON} src/build_eis_time_domain_priors.py \
  --ecm_dir "{ECM_SOURCE_DIR}" \
  --out_csv "{PRIOR_CSV}"'''
run(cmd, log_name='03_build_priors.log')
show_log('03_build_priors.log', n=120)

priors = pd.read_csv(PRIOR_CSV)
priors.head()


## 4. Time-domain ECM fitting from raw device data

This step uses raw `current / voltage / time` signals to extract device-side ECM-inspired features.

Common modes:

- `prior_mode = global`: ignore per-battery serial matching and use global prior bounds
- `prior_mode = serial_cycle`: use serial + cycle aligned priors when available
- `TD_CYCLE_MODE = best`: prefer the cycle whose relaxation segment has the most usable points
- `TD_SEGMENT_CHOICE = best_td`: prefer the segment with the best point-count / duration tradeoff


In [ ]:
RAW_DIR_FOR_FIT = RAW_SUBSET_DIR if RAW_SUBSET_DIR.exists() else RAW_DATA_DIR

cmd = f'''{VENV_PYTHON} src/extract_device_ecm_features.py   --raw_dir "{RAW_DIR_FOR_FIT}"   --eis_prior_csv "{PRIOR_CSV}"   --prior_mode global   --prior_align_mode last_le   --fit_mode td_only   --cycle_mode "{TD_CYCLE_MODE}"   --segment_choice "{TD_SEGMENT_CHOICE}"   --out_csv "{RAW_FEATURE_CSV}"'''
run(cmd, log_name='04_extract_device_ecm.log')
show_log('04_extract_device_ecm.log', n=120)

device_df = pd.read_csv(RAW_FEATURE_CSV)
device_df.head()


## 5. Reproduce the merged HYCL table used upstream of `latest_test_26_5_14_6`

This section rebuilds the merged table instead of reusing the final `14_6` training CSV.
It follows the same merge recipe as `latest_test_26_5_14_3`:

1. use the one-file-per-serial HYCL raw subset created above
2. use td-compatible priors built from the pre15 seed ECM directory
3. merge device features into the base workbook feature table
4. keep only HYCL rows

The resulting file should align with:

- `data/latest_test_26_5_14_3/feature_table_hycl_only_with_device_global.csv`


In [ ]:
cmd = f'''{VENV_PYTHON} src/extract_device_ecm_features.py \
  --raw_dir "{RAW_SUBSET_DIR}" \
  --eis_prior_csv "{PRIOR_CSV}" \
  --prior_mode global \
  --prior_align_mode last_le \
  --fit_mode td_only \
  --cycle_mode "{TD_CYCLE_MODE}" \
  --segment_choice "{TD_SEGMENT_CHOICE}" \
  --out_csv "{RAW_FEATURE_CSV}" \
  --feature_table_csv "{BASE_FEATURE_TABLE}" \
  --out_feature_table_csv "{MERGED_FEATURE_TABLE_ALL}" \
  --align_mode last_le'''
run(cmd, log_name='06_merge_device_features.log')
show_log('06_merge_device_features.log', n=120)

merged_all_df = pd.read_csv(MERGED_FEATURE_TABLE_ALL)
merged_df = merged_all_df[merged_all_df['group_tag'] == 'HYCL'].copy()
merged_df.to_csv(MERGED_FEATURE_TABLE, index=False)
print('saved', MERGED_FEATURE_TABLE)
print('all-shape =', merged_all_df.shape)
print('hycl-shape =', merged_df.shape)
print('n_cell_keys =', merged_df['cell_key'].nunique() if 'cell_key' in merged_df.columns else 'n/a')
merged_df.head()


In [ ]:
need = [
    'feat_cycle_t',
    'feat_capacity_t',
    'feat_capacity_slope_10',
    'feat_dcir_soc_t',
    'feat_dev_r0_proxy_ohm',
    'feat_dev_td_R_diff_total_ohm',
    'feat_dev_td_R_total_proxy_ohm',
]

train_df = merged_df.dropna(subset=need).copy()
train_df.to_csv(TRAIN_TABLE, index=False)
print('saved', TRAIN_TABLE)
print('merged_hycl_shape =', merged_df.shape)
print('train_shape =', train_df.shape)
for c in need:
    print(c, int(train_df[c].notna().sum()))
print('n_serials =', train_df['serial'].nunique() if 'serial' in train_df.columns else 'n/a')
print('n_cell_keys =', train_df['cell_key'].nunique() if 'cell_key' in train_df.columns else 'n/a')


### 5a. Final training table for the `14_6` reproduction

This matches the original `latest_test_26_5_14_6` training recipe:

- start from the rebuilt HYCL merged table produced above
- keep `cycle / capacity / capacity_slope / dcir`
- replace lab-side ECM model inputs with time-domain device-side ECM features
- require all seven selected training features to be non-null


In [ ]:
pd.read_csv(TRAIN_TABLE).head()


## 6. Model training

This section trains a classical XGBoost model. You can adapt the same pattern for MLP/CNN/LSTM or Transformer models.

In [ ]:
CUSTOM_FEATURES = ','.join([
    'feat_cycle_t',
    'feat_capacity_t',
    'feat_capacity_slope_10',
    'feat_dcir_soc_t',
    'feat_dev_r0_proxy_ohm',
    'feat_dev_td_R_diff_total_ohm',
    'feat_dev_td_R_total_proxy_ohm',
])

cmd = f'''{VENV_PYTHON} src/train_swelling_models.py \
  --table_csv "{TRAIN_TABLE}" \
  --out_dir "{MODEL_OUT_DIR}" \
  --target_mode current \
  --sample_mode rowwise \
  --label_mode absolute \
  --target_transform log \
  --max_input_cycle 120 \
  --models XGBoost \
  --feature_set custom \
  --custom_features "{CUSTOM_FEATURES}" \
  --xgb_n_estimators 1200 \
  --xgb_max_depth 4 \
  --xgb_learning_rate 0.015 \
  --xgb_subsample 0.85 \
  --xgb_colsample_bytree 0.85 \
  --xgb_min_child_weight 2 \
  --xgb_reg_alpha 0.05 \
  --xgb_reg_lambda 2.0 \
  --run_tag {RESULT_RUN_TAG} \
  --log_file "{LOG_DIR / '07_train_xgb.internal.log'}"'''
run(cmd, log_name='07_train_xgb.log')
show_log('07_train_xgb.log', n=120)


In [ ]:
result_csv = MODEL_OUT_DIR / f'results__current__absolute__current_cycle__{RESULT_RUN_TAG}.csv'
pred_csv = MODEL_OUT_DIR / f'predictions__current__absolute__current_cycle__{RESULT_RUN_TAG}.csv'
print('result_csv =', result_csv)
print('pred_csv   =', pred_csv)
if result_csv.exists():
    pd.read_csv(result_csv)


## 7. Evaluation and plotting

This step generates:

- prediction vs truth scatter
- permutation importance
- correlation matrix

using the bundled feature-analysis runner.

In [ ]:
cmd = f'''{VENV_PYTHON} src/run_feature_analysis.py \
  --table_csv "{TRAIN_TABLE}" \
  --results_dir "{MODEL_OUT_DIR}" \
  --out_dir "{MODEL_OUT_DIR}" \
  --custom_features "{CUSTOM_FEATURES}" \
  --group_tag HYCL \
  --target_mode current \
  --sample_mode rowwise \
  --label_mode absolute \
  --target_transform log \
  --max_input_cycle 120 \
  --model XGBoost \
  --metric mae \
  --n_repeats 30 \
  --corr_method spearman \
  --xgb_n_estimators 1200 \
  --xgb_max_depth 4 \
  --xgb_learning_rate 0.015 \
  --xgb_subsample 0.85 \
  --xgb_colsample_bytree 0.85 \
  --xgb_min_child_weight 2 \
  --xgb_reg_alpha 0.05 \
  --xgb_reg_lambda 2.0'''
run(cmd, log_name='08_feature_analysis.log')
show_log('08_feature_analysis.log', n=120)


In [ ]:
from IPython.display import Image, display

for name in [
    'pred_scatter__combined.png',
    'feature_corr__features_targets.png',
    'perm_importance__current__absolute__current_cycle__HYCL__XGBoost__mae.png',
]:
    p = MODEL_OUT_DIR / name
    if p.exists():
        print(p)
        display(Image(filename=str(p)))

## 8. Frequency-domain and time-domain ECM range plots

This section generates matched range plots for the five resistance variables used in the
frequency-domain td-compatible branch:

- `R0`
- `Rsei`
- `Rct`
- `Rw1`
- `Rw2`

Outputs are saved into `MODEL_OUT_DIR` as:

- `ecm_range__frequency_domain_prior_bounds__matched5.png`
- `ecm_range__time_domain_fitted_ranges__matched5.png`
- `ecm_range__freq_vs_time__matched5.png`
- plus the corresponding `.csv` summaries


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

priors = pd.read_csv(PRIOR_CSV)
train_df = pd.read_csv(TRAIN_TABLE)

matched5 = [
    ('R0', 'prior_R0_lb', 'prior_R0_ub', 'feat_dev_r0_abs_proxy_ohm'),
    ('Rsei', 'prior_Rsei_lb', 'prior_Rsei_ub', 'feat_dev_td_Rsei_ohm'),
    ('Rct', 'prior_Rct_lb', 'prior_Rct_ub', 'feat_dev_td_Rct_ohm'),
    ('Rw1', 'prior_Rw1_lb', 'prior_Rw1_ub', 'feat_dev_td_Rw1_ohm'),
    ('Rw2', 'prior_Rw2_lb', 'prior_Rw2_ub', 'feat_dev_td_Rw2_ohm'),
]

freq_rows = []
time_rows = []
for label, lb_col, ub_col, td_col in matched5:
    freq_lb = pd.to_numeric(priors.get(lb_col), errors='coerce')
    freq_ub = pd.to_numeric(priors.get(ub_col), errors='coerce')
    freq_rows.append({
        'parameter': label,
        'lb': float(freq_lb.min()) if freq_lb.notna().any() else np.nan,
        'ub': float(freq_ub.max()) if freq_ub.notna().any() else np.nan,
        'status': 'ok' if freq_lb.notna().any() and freq_ub.notna().any() else 'missing',
    })

    if td_col is None:
        time_rows.append({'parameter': label, 'lb': np.nan, 'ub': np.nan, 'status': 'not_fitted_in_current_impl'})
    else:
        td_vals = pd.to_numeric(train_df.get(td_col), errors='coerce')
        if td_vals.notna().any():
            time_rows.append({
                'parameter': label,
                'lb': float(td_vals.min()),
                'ub': float(td_vals.max()),
                'status': 'ok',
            })
        else:
            status = 'no_valid_fit'
            if 'feat_dev_td_fit_status' in train_df.columns and train_df['feat_dev_td_fit_status'].notna().any():
                status = str(train_df['feat_dev_td_fit_status'].dropna().iloc[0])
            time_rows.append({'parameter': label, 'lb': np.nan, 'ub': np.nan, 'status': status})

freq_df = pd.DataFrame(freq_rows)
time_df = pd.DataFrame(time_rows)

freq_csv = MODEL_OUT_DIR / 'ecm_range__frequency_domain_prior_bounds__matched5.csv'
time_csv = MODEL_OUT_DIR / 'ecm_range__time_domain_fitted_ranges__matched5.csv'
freq_df.to_csv(freq_csv, index=False)
time_df.to_csv(time_csv, index=False)
print('saved', freq_csv)
print('saved', time_csv)


def _plot_range_df(df, title, out_path, color):
    fig, ax = plt.subplots(figsize=(8, 4.8))
    y = np.arange(len(df))
    any_range = False
    for i, row in df.iterrows():
        lb = row['lb']
        ub = row['ub']
        if pd.notna(lb) and pd.notna(ub):
            any_range = True
            left = min(lb, ub)
            width = abs(ub - lb)
            if width == 0:
                width = 1e-12
            ax.barh(i, width, left=left, height=0.55, color=color, alpha=0.8)
            ax.text(ub, i, f' [{lb:.4g}, {ub:.4g}]', va='center', ha='left', fontsize=9)
        else:
            ax.text(0.02, i, f'N/A ({row["status"]})', va='center', ha='left', fontsize=9,
                    transform=ax.get_yaxis_transform())
    ax.set_yticks(y)
    ax.set_yticklabels(df['parameter'])
    ax.set_title(title)
    ax.set_xlabel('Resistance (ohm)')
    ax.grid(axis='x', alpha=0.25)
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(out_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    print('saved', out_path)


def _plot_compare(freq_df, time_df, out_path):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
    for ax, df, title, color in [
        (axes[0], freq_df, 'Frequency-domain prior bounds (matched5)', '#4c78a8'),
        (axes[1], time_df, 'Time-domain fitted ranges (matched5)', '#f58518'),
    ]:
        y = np.arange(len(df))
        for i, row in df.iterrows():
            lb = row['lb']
            ub = row['ub']
            if pd.notna(lb) and pd.notna(ub):
                left = min(lb, ub)
                width = abs(ub - lb)
                if width == 0:
                    width = 1e-12
                ax.barh(i, width, left=left, height=0.55, color=color, alpha=0.8)
                ax.text(ub, i, f' [{lb:.4g}, {ub:.4g}]', va='center', ha='left', fontsize=8)
            else:
                ax.text(0.02, i, 'N/A', va='center', ha='left', fontsize=9,
                        transform=ax.get_yaxis_transform())
        ax.set_title(title)
        ax.set_xlabel('Resistance (ohm)')
        ax.grid(axis='x', alpha=0.25)
        ax.set_yticks(y)
        ax.set_yticklabels(df['parameter'])
        ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(out_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    print('saved', out_path)

freq_png = MODEL_OUT_DIR / 'ecm_range__frequency_domain_prior_bounds__matched5.png'
time_png = MODEL_OUT_DIR / 'ecm_range__time_domain_fitted_ranges__matched5.png'
compare_png = MODEL_OUT_DIR / 'ecm_range__freq_vs_time__matched5.png'

_plot_range_df(freq_df, 'Frequency-domain prior bounds (matched5)', freq_png, '#4c78a8')
_plot_range_df(time_df, 'Time-domain fitted ranges (matched5)', time_png, '#f58518')
_plot_compare(freq_df, time_df, compare_png)

for p in [freq_png, time_png, compare_png]:
    if p.exists():
        print(p)
        display(Image(filename=str(p)))


## 9. Final notes

Recommended things to record after each run:

- dataset paths used
- whether the run is hybrid or device-oriented
- prior mode (`global`, `serial_cycle`, etc.)
- final feature list
- final `MAE / RMSE`
- key plots
- any failure modes such as `insufficient_points`